In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import os
from tokenizers import Tokenizer

# ==========================================
# 1. THE ARCHITECTURE (Must be defined first)
# ==========================================
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout_prob):
        super().__init__()
        self.c_attn = nn.Linear(d_model, 3 * d_model, bias=False)
        self.c_proj = nn.Linear(d_model, d_model, bias=False)
        self.n_heads = n_heads
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.d_model, dim=2)
        k = k.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        q = q.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        v = v.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.dropout(self.c_proj(y))

class FeedForward(nn.Module):
    def __init__(self, d_model, dropout_prob):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout_prob),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, d_model, n_heads, dropout_prob):
        super().__init__()
        self.ln_1, self.ln_2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout_prob)
        self.mlp = FeedForward(d_model, dropout_prob)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config['vocab_size'], config['d_model']),
            wpe = nn.Embedding(config['max_seq_len'], config['d_model']),
            drop = nn.Dropout(config['dropout_prob']),
            h = nn.ModuleList([Block(config['d_model'], config['n_heads'], config['dropout_prob']) for _ in range(config['n_layers'])]),
            ln_f = nn.LayerNorm(config['d_model']),
        ))
        self.lm_head = nn.Linear(config['d_model'], config['vocab_size'], bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h: x = block(x)
        logits = self.lm_head(self.transformer.ln_f(x))
        return logits, None

# ==========================================
# 2. LOADING & INFERENCE
# ==========================================
TOKENIZER_DIR = "/kaggle/input/notebooks/codingmaster24/tokenizer"
MODEL_DIR = "/kaggle/input/notebooks/codingmaster24/gpu-model-traning"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = Tokenizer.from_file(os.path.join(TOKENIZER_DIR, "tokenizer.json"))
with open(os.path.join(MODEL_DIR, "config.json"), "r") as f:
    config = json.load(f)

print("Initializing model architecture...")
model = GPT(config).to(device)

print("Loading weights...")
state_dict = torch.load(os.path.join(MODEL_DIR, "gpt_10m_model.pt"), map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.eval()
print("Model ready!")

# ==========================================
# 3. GENERATION WITH BULLETPROOF DECODING
# ==========================================
def generate(prompt, max_len=60, temp=0.7):
    input_ids = torch.tensor(tokenizer.encode(prompt).ids, device=device).unsqueeze(0)
    
    model.eval()
    with torch.no_grad():
        for _ in range(max_len):
            idx_cond = input_ids[:, -config['max_seq_len']:]
            logits, _ = model(idx_cond)
            
            logits = logits[:, -1, :] / temp
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            input_ids = torch.cat((input_ids, next_token), dim=1)
            
            if next_token.item() == tokenizer.token_to_id("[SEP]"):
                break

    # --- THE FIX: Bulletproof Manual Decoding ---
    generated_list = input_ids[0].tolist()
    
    # 1. Convert IDs to raw token strings
    raw_tokens = []
    for token_id in generated_list:
        token_str = tokenizer.id_to_token(token_id)
        if token_str is not None:
            raw_tokens.append(token_str)
            
    # 2. Join tightly with NO spaces
    raw_text = "".join(raw_tokens)
    
    # 3. Replace hidden Tokenizer space-markers with actual human spaces
    clean_text = raw_text.replace("Ġ", " ")   # Fixes GPT-2 BPE
    clean_text = clean_text.replace(" ", " ") # Fixes SentencePiece
    clean_text = clean_text.replace("##", "") # Fixes BERT WordPiece
    clean_text = clean_text.replace("</w>", " ") # Fixes legacy BPE
    
    # 4. Strip out any meta-tags
    clean_text = clean_text.replace("[SEP]", "").replace("[PAD]", "").replace("[UNK]", "")
    
    return clean_text.strip()

# Run the test
print("\n--- Fixed Output ---")
print(generate("What is data science", temp=0.4))

Initializing model architecture...
Loading weights...
Model ready!

--- Fixed Output ---
Whatisdatascience.Theywasaproblemsthattheysaystheonlyandproblemsandsometimeth


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import os
from tokenizers import Tokenizer

# ==========================================
# 1. THE ARCHITECTURE (Must be defined first)
# ==========================================
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout_prob):
        super().__init__()
        self.c_attn = nn.Linear(d_model, 3 * d_model, bias=False)
        self.c_proj = nn.Linear(d_model, d_model, bias=False)
        self.n_heads = n_heads
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.d_model, dim=2)
        k = k.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        q = q.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        v = v.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.dropout(self.c_proj(y))

class FeedForward(nn.Module):
    def __init__(self, d_model, dropout_prob):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout_prob),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, d_model, n_heads, dropout_prob):
        super().__init__()
        self.ln_1, self.ln_2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout_prob)
        self.mlp = FeedForward(d_model, dropout_prob)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config['vocab_size'], config['d_model']),
            wpe = nn.Embedding(config['max_seq_len'], config['d_model']),
            drop = nn.Dropout(config['dropout_prob']),
            h = nn.ModuleList([Block(config['d_model'], config['n_heads'], config['dropout_prob']) for _ in range(config['n_layers'])]),
            ln_f = nn.LayerNorm(config['d_model']),
        ))
        self.lm_head = nn.Linear(config['d_model'], config['vocab_size'], bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h: x = block(x)
        logits = self.lm_head(self.transformer.ln_f(x))
        return logits, None

# ==========================================
# 2. LOADING & INFERENCE (Using your paths)
# ==========================================
TOKENIZER_DIR = "/kaggle/input/notebooks/codingmaster24/tokenizer"
MODEL_DIR = "/kaggle/input/notebooks/codingmaster24/gpu-model-traning"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Tokenizer and Config
tokenizer = Tokenizer.from_file(os.path.join(TOKENIZER_DIR, "tokenizer.json"))
with open(os.path.join(MODEL_DIR, "config.json"), "r") as f:
    config = json.load(f)

print("Initializing model architecture...")
model = GPT(config).to(device)

print("Loading weights...")
state_dict = torch.load(os.path.join(MODEL_DIR, "gpt_10m_model.pt"), map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.eval()
print("Model ready!")

# ==========================================
# 3. GENERATION
# ==========================================
def generate(prompt, max_len=60, temp=0.7):
    # 1. Encode the prompt correctly
    # .ids returns the actual integers the model understands
    input_ids = torch.tensor(tokenizer.encode(prompt).ids, device=device).unsqueeze(0)
    
    model.eval()
    with torch.no_grad():
        for _ in range(max_len):
            idx_cond = input_ids[:, -config['max_seq_len']:]
            logits, _ = model(idx_cond)
            
            # Apply temperature and sample
            logits = logits[:, -1, :] / temp
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Append to sequence
            input_ids = torch.cat((input_ids, next_token), dim=1)
            
            # Stop if the model generates an End of Sentence token
            if next_token.item() == tokenizer.token_to_id("[SEP]"):
                break

    # 2. THE FIX: Proper decoding
    generated_list = input_ids[0].tolist()
    
    # We use skip_special_tokens=True to remove [PAD], [UNK], etc.
    # The tokenizer.decode method should handle the 'Ġ' characters automatically
    clean_text = tokenizer.decode(generated_list, skip_special_tokens=True)
    
    return clean_text

# Run the test
print("\n--- Fixed Output ---")
print(generate("In the field of data science", temp=0.4))

Initializing model architecture...
Loading weights...
Model ready!

--- Fixed Output ---
I n t h e f i e l d o f d a t a s c i e n c e . T h e r e i s n o t b e d e l e t e d t h e p a r t y o f t h e s e c o n d s t o r y . I n t h e p r o c e s s i o n


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import os
from tokenizers import Tokenizer

# ==========================================
# 1. THE ARCHITECTURE (The "Body")
# ==========================================
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout_prob):
        super().__init__()
        self.c_attn = nn.Linear(d_model, 3 * d_model, bias=False)
        self.c_proj = nn.Linear(d_model, d_model, bias=False)
        self.n_heads = n_heads
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.d_model, dim=2)
        k = k.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        q = q.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        v = v.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.dropout(self.c_proj(y))

class FeedForward(nn.Module):
    def __init__(self, d_model, dropout_prob):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout_prob),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, d_model, n_heads, dropout_prob):
        super().__init__()
        self.ln_1, self.ln_2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout_prob)
        self.mlp = FeedForward(d_model, dropout_prob)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config['vocab_size'], config['d_model']),
            wpe = nn.Embedding(config['max_seq_len'], config['d_model']),
            drop = nn.Dropout(config['dropout_prob']),
            h = nn.ModuleList([Block(config['d_model'], config['n_heads'], config['dropout_prob']) for _ in range(config['n_layers'])]),
            ln_f = nn.LayerNorm(config['d_model']),
        ))
        self.lm_head = nn.Linear(config['d_model'], config['vocab_size'], bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h: x = block(x)
        logits = self.lm_head(self.transformer.ln_f(x))
        return logits, None

# ==========================================
# 2. LOADING & INFERENCE
# ==========================================
TOKENIZER_DIR = "/kaggle/input/notebooks/codingmaster24/tokenizer"
MODEL_DIR = "/kaggle/input/notebooks/codingmaster24/gpu-model-traning"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = Tokenizer.from_file(os.path.join(TOKENIZER_DIR, "tokenizer.json"))
with open(os.path.join(MODEL_DIR, "config.json"), "r") as f:
    config = json.load(f)

print("Initializing and loading model...")
model = GPT(config).to(device)
state_dict = torch.load(os.path.join(MODEL_DIR, "gpt_10m_model.pt"), map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.eval()

def generate(prompt, max_len=50):
    input_ids = torch.tensor(tokenizer.encode(prompt).ids, device=device).unsqueeze(0)
    for _ in range(max_len):
        logits, _ = model(input_ids[:, -config['max_seq_len']:])
        next_token = torch.multinomial(F.softmax(logits[:, -1, :] / 0.7, dim=-1), num_samples=1)
        input_ids = torch.cat((input_ids, next_token), dim=1)
    return tokenizer.decode(input_ids[0].tolist())

# Test it!
print(generate("Once upon a time in science"))

Initializing and loading model...
O n c e u p o n a t i m e i n s c i e n c e . T h e r e i s n o t a g r o u p o n a l s t u d y o r e v e n t h e p o s i t i o n o f g r e a t


In [ ]:
import torch
import torch.nn.functional as F
import json
from tokenizers import Tokenizer # Assuming you used the HF tokenizers library

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load the Tokenizer
# Update this path if your tokenizer is somewhere else!
tokenizer_path = "/kaggle/input/notebooks/codingmaster24/tokenizer/tokenizer.json"
tokenizer = Tokenizer.from_file(tokenizer_path)

# 2. Load the Config and Model
config_path = "/kaggle/working/config.json"
with open(config_path, "r") as f:
    config = json.load(f)

print("Loading model architecture...")
model = GPT(config) # This requires your GPT class to be defined above

print("Loading saved weights...")
# We use map_location to ensure it loads smoothly whether on CPU or GPU
state_dict = torch.load("/kaggle/working/gpt_10m_model.pt", map_location=device, weights_only=True)
model.load_state_dict(state_dict)

model.to(device)
model.eval() # CRITICAL: Sets dropout to 0 and preps for generation
print("Model ready!")

# 3. The Generation Function
def ask_model(prompt_text, max_new_tokens=50, temperature=0.8, top_k=50):
    # Convert your text prompt to token IDs
    input_ids = tokenizer.encode(prompt_text).ids
    x = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0) # Shape: (1, seq_len)
    
    print(f"\n--- Generating response for: '{prompt_text}' ---")
    
    with torch.no_grad(): # Don't track gradients during inference to save RAM
        for _ in range(max_new_tokens):
            # Crop context to max_seq_len if it gets too long
            if x.size(1) > config['max_seq_len']:
                x = x[:, -config['max_seq_len']:]
                
            # Forward pass
            logits, _ = model(x)
            
            # We only care about the prediction for the VERY LAST token
            next_token_logits = logits[:, -1, :] / temperature
            
            # Top-K filtering (prevents the model from picking completely random garbage words)
            v, _ = torch.topk(next_token_logits, min(top_k, next_token_logits.size(-1)))
            next_token_logits[next_token_logits < v[:, [-1]]] = -float('Inf')
            
            # Convert logits to probabilities
            probs = F.softmax(next_token_logits, dim=-1)
            
            # Sample the next token ID from the probability distribution
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Append it to the sequence and loop
            x = torch.cat((x, next_token), dim=1)
            
    # Convert the final list of IDs back into readable text
    generated_ids = x[0].tolist()
    final_text = tokenizer.decode(generated_ids)
    
    return final_text

# ==========================================
# TEST YOUR MODEL HERE
# ==========================================
my_prompt = "The future of artificial intelligence is"
response = ask_model(my_prompt, max_new_tokens=100, temperature=0.7)
print(response)